# PT02: Logistic regression in PyTorch — solutions

Given a few measurements of a penguin, can we predict its species? We will build a linear classifier and follow its training through to evaluation.

Read PT01 before this session. We will use its tensor operations, `backward()`, and parameter updates. The data loading and plotting code are provided. There are four activities: the forward pass, the loss, the training step, and an investigation of a training run. We then compare learning rates and extend training to mini-batches.

This is the solutions notebook. Reference implementations are included directly, and unfinished activities fall back to them so that the full notebook remains runnable. The student notebook is distributed separately without these cells.

> Try to avoid AI coding assistants for these simple exercises.

We use `jaxtyping` to annotate shapes at function boundaries. For example, `Float[torch.Tensor, "batch features"]` means a floating-point tensor with two named axes. These annotations document the expected inputs and outputs; they do not enable runtime checking on their own. You still need to work out the intermediate shapes.

If the import fails, install `jaxtyping` with `%pip install jaxtyping` in a code cell, then rerun the imports.

In [ ]:
from pathlib import Path
import copy
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from jaxtyping import Float, Int
from typing import Optional
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

torch.manual_seed(7)
torch.set_printoptions(precision=3, sci_mode=False)
print("PyTorch", torch.__version__)

## 1. The data

We use the [Palmer Penguins dataset](https://allisonhorst.github.io/palmerpenguins/), with four numeric measurements and three species: Adelie, Chinstrap and Gentoo. Each row corresponds to a penguin. We omit the two rows with missing measurements.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"
DATA_SHA256 = "f204db2c753b0937caac3cb35258562c14f073e4bbc76be24b4c51ce22767a93"
data_path = Path("data/penguins.csv")
if not data_path.exists():
    from urllib.request import urlopen
    data_path.parent.mkdir(parents=True, exist_ok=True)
    data_path.write_bytes(urlopen(DATA_URL, timeout=30).read())
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == DATA_SHA256, "Dataset changed; use the supplied data/penguins.csv."

features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
classes = ["Adelie", "Chinstrap", "Gentoo"]
penguins = pd.read_csv(data_path).dropna(subset=features + ["species"])
X = torch.tensor(penguins[features].to_numpy(), dtype=torch.float32)
y = torch.tensor(pd.Categorical(penguins.species, categories=classes).codes.copy(), dtype=torch.long)
assert X.shape == (342, 4) and y.shape == (342,)
penguins[features + ["species"]].head()

We split the data before estimating any means or scales. Training uses 60% of the examples, validation roughly 20%, and the remaining examples are held out for the final test. Stratification preserves approximately the same class proportions in each split.

In [ ]:
indices = np.arange(len(y))
train_idx, rest_idx = train_test_split(indices, test_size=0.4, stratify=y.numpy(), random_state=7)
val_idx, test_idx = train_test_split(rest_idx, test_size=0.5, stratify=y[rest_idx].numpy(), random_state=7)

Xtrain_raw, ytrain = X[train_idx], y[train_idx]
Xval_raw, yval = X[val_idx], y[val_idx]
Xtest_raw, ytest = X[test_idx], y[test_idx]
mean = Xtrain_raw.mean(dim=0)
scale = Xtrain_raw.std(dim=0, correction=0).clamp_min(1e-8)
Xtrain = (Xtrain_raw - mean) / scale
Xval = (Xval_raw - mean) / scale
Xtest = (Xtest_raw - mean) / scale

print("Examples:", len(ytrain), "training;", len(yval), "validation;", len(ytest), "test")
print("Training means:", Xtrain.mean(dim=0))
torch.testing.assert_close(Xtrain.mean(dim=0), torch.zeros(4), atol=2e-6, rtol=0)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for c, name in enumerate(classes):
    points = Xtrain_raw[ytrain == c]
    ax.scatter(points[:, 0], points[:, 1], label=name, marker=["o", "^", "s"][c], s=24, alpha=0.8)
ax.set(xlabel="Bill length (mm)", ylabel="Bill depth (mm)")
ax.legend()
fig.tight_layout()
plt.show()

## 2. The model

Recall the batched linear model

$$Z=XW^\top+b,$$

with $X\sim(B,D)$, $W\sim(C,D)$, $b\sim(C,)$ and $Z\sim(B,C)$. Here $D=4$ and $C=3$. The outputs are logits; softmax is needed only when we want probabilities.

### Activity 1: a batched forward pass

Implement `linear_logits` using tensor operations. Check it on the small example below, then explain how the bias is applied to every example. The model class is provided so we can concentrate on the computation.

In [ ]:
def linear_logits(X: Float[torch.Tensor, "batch features"], W: Float[torch.Tensor, "classes features"], b: Float[torch.Tensor, "classes"]) -> Optional[Float[torch.Tensor, "batch classes"]]:
    # Replace None with your batched computation.
    return None

In [ ]:
# Reference used only when linear_logits returns None.
def reference_linear_logits(X: Float[torch.Tensor, "batch features"], W: Float[torch.Tensor, "classes features"], b: Float[torch.Tensor, "classes"]) -> Float[torch.Tensor, "batch classes"]:
    return X @ W.T + b

def forward_logits(X: Float[torch.Tensor, "batch features"], W: Float[torch.Tensor, "classes features"], b: Float[torch.Tensor, "classes"]) -> Float[torch.Tensor, "batch classes"]:
    result = linear_logits(X, W, b)
    return reference_linear_logits(X, W, b) if result is None else result

In [ ]:
small_X = torch.tensor([[1., 2.], [3., 4.]])
small_W = torch.tensor([[1., 0.], [0., 1.], [-1., 1.]])
small_b = torch.tensor([0.5, -0.5, 1.])
expected = torch.tensor([[1.5, 1.5, 2.], [3.5, 3.5, 2.]])
torch.testing.assert_close(forward_logits(small_X, small_W, small_b), expected)
print("Using", "reference" if linear_logits(small_X, small_W, small_b) is None else "your", "forward pass")

`nn.Module` collects the parameters of a model. Wrapping a tensor in `nn.Parameter` registers it as a parameter and enables gradients by default. We use zero initialization here: for a linear classifier this is fine. The same choice would need reconsideration for hidden units in an MLP.

In [ ]:
class LogisticRegression(torch.nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.weight = torch.nn.Parameter(torch.zeros(output_dim, input_dim))
        self.bias = torch.nn.Parameter(torch.zeros(output_dim))

    def forward(self, X: Float[torch.Tensor, "batch features"]) -> Float[torch.Tensor, "batch classes"]:
        return forward_logits(X, self.weight, self.bias)

model = LogisticRegression(Xtrain.shape[1], len(classes))
assert model(Xtrain[:5]).shape == (5, 3)
print("Parameter tensors:", len(list(model.parameters())))
print("Trainable scalar parameters:", sum(p.numel() for p in model.parameters()))

## 3. Loss and accuracy

For logits $z$ and class index $y$, cross-entropy can be written as

$$\ell(z,y)=\log\sum_c\exp(z_c)-z_y.$$

### Activity 2: the loss

Implement `student_cross_entropy` from the formula above. Use `torch.logsumexp` for the normalization term, select the correct-class logit from each row, and average over examples. Do not call `F.cross_entropy` inside your implementation; we will use it to check the result.

Before coding, give the shapes of the normalization term, the selected logits and the final loss. Accuracy is supplied.

In [ ]:
def student_cross_entropy(logits: Float[torch.Tensor, "batch classes"], targets: Int[torch.Tensor, "batch"]) -> Optional[Float[torch.Tensor, ""]]:
    # Replace None with the batched loss.
    return None

In [ ]:
# Reference used only while the loss activity is unfilled.
def reference_cross_entropy(logits: Float[torch.Tensor, "batch classes"], targets: Int[torch.Tensor, "batch"]) -> Float[torch.Tensor, ""]:
    rows = torch.arange(len(targets), device=targets.device)
    return (torch.logsumexp(logits, dim=1) - logits[rows, targets]).mean()

def cross_entropy(logits: Float[torch.Tensor, "batch classes"], targets: Int[torch.Tensor, "batch"]) -> Float[torch.Tensor, ""]:
    result = student_cross_entropy(logits, targets)
    return reference_cross_entropy(logits, targets) if result is None else result


In [ ]:
def accuracy(logits: Float[torch.Tensor, "batch classes"], targets: Int[torch.Tensor, "batch"]) -> Float[torch.Tensor, ""]:
    return (logits.argmax(dim=1) == targets).float().mean()

logits = model(Xtrain)
print("Initial loss:", cross_entropy(logits, ytrain).item())
print("Initial accuracy:", accuracy(logits, ytrain).item())
# Zero logits give uniform probabilities and loss log(C).
torch.testing.assert_close(cross_entropy(logits, ytrain), torch.tensor(float(np.log(3))))
print("Using", "reference" if student_cross_entropy(logits, ytrain) is None else "your", "loss")

At initialization all logits are zero. The probabilities are uniform, but `argmax` resolves a tie by selecting the first class. Initial accuracy is therefore the fraction of that class, not necessarily one third.

Compare both the loss and its gradients with PyTorch. We also use large logits, where a direct `log(softmax(...))` can lose numerical stability.

In [ ]:
# Check unequal losses as well as extreme logits: averaging must be over examples.
for values, labels in [([[2., -1., 0.], [0., 1., -2.]], [0, 2]),
                       ([[1000., 0., -1000.], [-1000., 0., 1000.]], [2, 0])]:
    probe = torch.tensor(values, requires_grad=True)
    targets = torch.tensor(labels)
    ours = cross_entropy(probe, targets)
    builtin = F.cross_entropy(probe, targets)
    torch.testing.assert_close(ours, builtin)
    g_ours, = torch.autograd.grad(ours, probe)
    g_builtin, = torch.autograd.grad(builtin, probe)
    torch.testing.assert_close(g_ours, g_builtin)
    assert torch.isfinite(ours)
print("Loss values and gradients agree.")

## 4. One training step

We first carry out one step explicitly. Notice the order: clear gradients, compute the loss, differentiate, update. The update runs under `no_grad`; the forward pass does not.

In [ ]:
model = LogisticRegression(Xtrain.shape[1], len(classes))
for parameter in model.parameters():
    parameter.grad = None
loss = cross_entropy(model(Xtrain), ytrain)
loss.backward()
print("Weight gradient shape:", model.weight.grad.shape)
print("Bias gradient shape:", model.bias.grad.shape)
with torch.no_grad():
    for parameter in model.parameters():
        parameter -= 0.1 * parameter.grad
print("Loss before:", loss.item())
print("Loss after:", cross_entropy(model(Xtrain), ytrain).item())

### Activity 3: train the classifier

Put the update into `training_step`. It should update the model in place and return the loss **before** the update as a Python number. Data are arguments, so the function can later receive a mini-batch without using global variables.

Run the checks and the complete training loop. Inspect the training and validation curves together. Keep the test split for the end; we will compare learning rates below.

In [ ]:
def training_step(model: torch.nn.Module, X: Float[torch.Tensor, "batch features"], y: Int[torch.Tensor, "batch"], lr: float) -> Optional[float]:
    # Replace None with your training step.
    return None

In [ ]:
def reference_training_step(model: torch.nn.Module, X: Float[torch.Tensor, "batch features"], y: Int[torch.Tensor, "batch"], lr: float) -> float:
    for parameter in model.parameters():
        parameter.grad = None
    loss = cross_entropy(model(X), y)
    loss.backward()
    with torch.no_grad():
        for parameter in model.parameters():
            parameter -= lr * parameter.grad
    return loss.item()

def run_step(model: torch.nn.Module, X: Float[torch.Tensor, "batch features"], y: Int[torch.Tensor, "batch"], lr: float) -> float:
    # A skipped exercise returns None without modifying the model.
    before = [p.detach().clone() for p in model.parameters()]
    result = training_step(model, X, y, lr)
    if result is None:
        assert all(torch.equal(p, saved) for p, saved in zip(model.parameters(), before)), "Your step changed parameters but returned None. Return loss.item()."
        # Clear any gradients left by a partially completed exercise.
        return reference_training_step(model, X, y, lr)
    return result

@torch.no_grad()
def evaluate(model, X, y):
    model.eval()
    logits = model(X)
    return cross_entropy(logits, y).item(), accuracy(logits, y).item()

In [ ]:
# Compare two consecutive updates: the second detects uncleared gradients.
check_model = LogisticRegression(4, 3)
reference_model = copy.deepcopy(check_model)
for _ in range(2):
    actual = run_step(check_model, Xtrain[:16], ytrain[:16], 0.1)
    expected_loss = reference_training_step(reference_model, Xtrain[:16], ytrain[:16], 0.1)
    assert isinstance(actual, float), "Return loss.item(), not a tensor."
    assert abs(actual - expected_loss) < 1e-6
    for parameter, expected_parameter in zip(check_model.parameters(), reference_model.parameters()):
        torch.testing.assert_close(parameter, expected_parameter)
print("Both updates agree with the reference.")

In [ ]:
def fit(model, Xtrain, ytrain, Xval, yval, lr=0.1, epochs=200):
    history = {name: [] for name in ["train_loss", "val_loss", "train_accuracy", "val_accuracy"]}
    for _ in range(epochs):
        model.train()
        run_step(model, Xtrain, ytrain, lr)
        # Both curves measure the model after the same update.
        train_loss, train_acc = evaluate(model, Xtrain, ytrain)
        val_loss, val_acc = evaluate(model, Xval, yval)
        for key, value in zip(history, [train_loss, val_loss, train_acc, val_acc]):
            history[key].append(value)
    return history

model = LogisticRegression(4, 3)
history = fit(model, Xtrain, ytrain, Xval, yval)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
epochs = range(1, len(history["train_loss"]) + 1)
for split, style in [("train", "-"), ("val", "--")]:
    axes[0].plot(epochs, history[f"{split}_loss"], style, label=split)
    axes[1].plot(epochs, history[f"{split}_accuracy"], style, label=split)
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy")
axes[1].set(xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1.03))
for ax in axes:
    ax.legend()
fig.tight_layout()
plt.show()
print("Final validation loss and accuracy:", evaluate(model, Xval, yval))

Here each update uses the full training set, so one update is also one epoch. For mini-batch training, an epoch contains several updates.

Loss can decrease while accuracy stays unchanged: the probabilities can improve without changing the largest logit. Look for an interval in your curves where this happens.

### Comparing learning rates

Before running this cell, predict how the curves will differ for `0.001`, `0.1` and `10.0`. Every run starts from the same parameters and receives 200 full-batch updates. Only the learning rate changes.

Compare the loss curves as well as the final validation values. Does the largest rate make faster progress throughout? Does accuracy tell the same story? Select a run using validation loss; do not inspect the test set yet.

In [ ]:
learning_rates = [0.001, 0.1, 10.0]
initial_model = LogisticRegression(4, 3)
lr_models, lr_histories = {}, {}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for rate, style in zip(learning_rates, ["-", "--", ":"]):
    candidate = copy.deepcopy(initial_model)
    trace = fit(candidate, Xtrain, ytrain, Xval, yval, lr=rate, epochs=200)
    lr_models[rate], lr_histories[rate] = candidate, trace
    for ax, split in zip(axes, ["train", "val"]):
        ax.plot(range(1, 201), trace[f"{split}_loss"], style, label=f"lr={rate:g}")
    print(f"lr={rate:g}: validation loss={trace['val_loss'][-1]:.4f}, accuracy={trace['val_accuracy'][-1]:.3f}")
for ax, title in zip(axes, ["Training", "Validation"]):
    ax.set(xlabel="Update", ylabel="Cross-entropy", title=title, yscale="log")
    ax.legend()
fig.tight_layout()
plt.show()
selected_lr = min(learning_rates, key=lambda rate: lr_histories[rate]["val_loss"][-1])
model = lr_models[selected_lr]
print("Selected learning rate:", selected_lr)

## 5. Investigating a training run

### Activity 4: a different training run

A colleague gives you the function below. It runs, and its loss may even decrease faster at first. Does it implement the same gradient descent procedure?

Compare it with your working step using identical initial parameters and data. Find a small experiment that distinguishes the two procedures. Write down what you expect, run it, then repair the function. A falling loss alone is not enough to decide.

In [ ]:
def suspect_step(model: torch.nn.Module, X: Float[torch.Tensor, "batch features"], y: Int[torch.Tensor, "batch"], lr: float) -> float:
    loss = cross_entropy(model(X), y)
    loss.backward()
    with torch.no_grad():
        for parameter in model.parameters():
            parameter -= lr * parameter.grad
    return loss.item()

normal = LogisticRegression(4, 3)
suspect = copy.deepcopy(normal)
normal_losses, suspect_losses = [], []
normal_norms, suspect_norms = [], []
for _ in range(30):
    run_step(normal, Xtrain, ytrain, 0.1)
    suspect_step(suspect, Xtrain, ytrain, 0.1)
    normal_losses.append(evaluate(normal, Xtrain, ytrain)[0])
    suspect_losses.append(evaluate(suspect, Xtrain, ytrain)[0])
    normal_norms.append(normal.weight.grad.norm().item())
    suspect_norms.append(suspect.weight.grad.norm().item())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
updates = range(1, len(normal_losses) + 1)
axes[0].plot(updates, normal_losses, label="Working step")
axes[0].plot(updates, suspect_losses, "--", label="Suspect step")
axes[0].set(xlabel="Update", ylabel="Training cross-entropy")
axes[1].plot(updates, normal_norms, label="Working step")
axes[1].plot(updates, suspect_norms, "--", label="Suspect step")
axes[1].set(xlabel="Update", ylabel="Norm of weight.grad")
for ax in axes:
    ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# What is wrong?

In [ ]:
# Reference probe: differentiate twice without updating the parameters.
probe_model = LogisticRegression(4, 3)
cross_entropy(probe_model(Xtrain), ytrain).backward()
first = probe_model.weight.grad.clone()
cross_entropy(probe_model(Xtrain), ytrain).backward()
torch.testing.assert_close(probe_model.weight.grad, 2 * first)
print("Same parameters, same data, twice the stored gradient.")
# Clear gradients at the start of suspect_step, as in reference_training_step.

<details>
<summary>Discussion after the investigation</summary>

The suspect step accumulates gradients from earlier steps. After parameters change, those earlier gradients are evaluated at different points. The update is therefore no longer the intended gradient descent step. It may still reduce the loss, which is why looking only at the loss curve can miss the problem.

Accumulation can be intentional when combining mini-batches before one update, with the loss scaling chosen appropriately. Here we update after every backward pass and never clear the gradients.

</details>

## 6. Using the standard components

We can now replace the explicit matrix parameters with `nn.Linear`, the loss with `F.cross_entropy`, and the update with an optimizer. These abstractions should implement the same computation.

To compare them, start with identical parameters. Comparing independently initialized models would mix implementation differences with initialization differences.

In [ ]:
manual = LogisticRegression(4, 3)
standard = torch.nn.Linear(4, 3)
with torch.no_grad():
    standard.weight.copy_(manual.weight)
    standard.bias.copy_(manual.bias)
optimizer = torch.optim.SGD(standard.parameters(), lr=0.1)

manual_loss = run_step(manual, Xtrain, ytrain, 0.1)
optimizer.zero_grad(set_to_none=True)
standard_loss = F.cross_entropy(standard(Xtrain), ytrain)
standard_loss.backward()
optimizer.step()

torch.testing.assert_close(standard.weight, manual.weight)
torch.testing.assert_close(standard.bias, manual.bias)
torch.testing.assert_close(standard(Xval), manual(Xval))
print("Manual and standard updates agree.")

## 7. From full batches to mini-batches

A `TensorDataset` pairs each feature row with its label. A `DataLoader` groups these pairs into batches and can shuffle their order each epoch.

Before running: with 205 examples and batches of 32, how many updates are there in one epoch? How many examples are in the final batch?

We start a separate model from the same initial parameters and use `lr=0.1`. Compare it with the full-batch run at that rate. Equal epoch counts mean equal passes through the data, but different numbers of parameter updates; this is not a comparison at equal update counts.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(Xtrain, ytrain), batch_size=32,
                          shuffle=True, drop_last=False,
                          generator=torch.Generator().manual_seed(7))
mini_model = copy.deepcopy(initial_model)
mini_history = {"train_loss": [], "val_loss": []}
mini_updates = 0
batch_sizes = []
for epoch in range(30):
    mini_model.train()
    for batch_X, batch_y in train_loader:
        run_step(mini_model, batch_X, batch_y, lr=0.1)
        mini_updates += 1
        if epoch == 0:
            batch_sizes.append(len(batch_y))
    # Evaluate the current model on each complete split. Do not average
    # unweighted batch losses: the final batch contains fewer examples.
    mini_history["train_loss"].append(evaluate(mini_model, Xtrain, ytrain)[0])
    mini_history["val_loss"].append(evaluate(mini_model, Xval, yval)[0])
print("Batch sizes in one epoch:", batch_sizes)
print("Epochs: 30; updates:", mini_updates)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, 31), mini_history["val_loss"], label="Mini-batch, 32 examples")
ax.plot(range(1, 31), lr_histories[0.1]["val_loss"][:30], "--", label="Full batch")
ax.set(xlabel="Epoch", ylabel="Validation cross-entropy")
ax.legend()
fig.tight_layout()
plt.show()

Why can two batches give different gradients at the same parameter values? What changes if you set `batch_size=len(ytrain)`? Check that one such update agrees with the full-batch update, starting from identical parameters.

## 8. Final evaluation

The model named `model` is the full-batch run selected by final validation loss in the learning-rate experiment and is evaluated here on the test split.

Compare its accuracy with always predicting the most common training class. The confusion matrix shows which species it confuses. If you use the test result to change the model again, it is no longer a final held-out evaluation.

In [ ]:
test_loss, test_accuracy = evaluate(model, Xtest, ytest)
majority_class = torch.bincount(ytrain).argmax()
baseline_accuracy = (ytest == majority_class).float().mean().item()
print(f"Test cross-entropy: {test_loss:.3f}")
print(f"Test accuracy: {test_accuracy:.3f}")
print(f"Majority-class accuracy: {baseline_accuracy:.3f}")

with torch.no_grad():
    predicted = model(Xtest).argmax(dim=1)
confusion = torch.bincount(ytest * 3 + predicted, minlength=9).reshape(3, 3)
confusion_table = pd.DataFrame(confusion.numpy(), index=classes, columns=classes)
confusion_table.index.name = "True species"
confusion_table.columns.name = "Predicted species"
confusion_table

Before leaving the notebook, pick one incorrect prediction and inspect its logits or probabilities. What can you conclude from them? What would require more evidence?

The same training procedure will work when we replace the linear model with an MLP. We will make that change after discussing hidden representations.

## References

Data: Horst, Hill and Gorman (2020), [palmerpenguins](https://allisonhorst.github.io/palmerpenguins/), CC0. Original data collected by Kristen Gorman and Palmer Station LTER; see the dataset page for the ecological study and data citations.

- [CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- [nn.Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)
- [Autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html)